# Healthcare Lab Test Demand Forecasting

Forecast monthly laboratory test demand using ETS and XGBoost with walk-forward model selection.

**Note:** This notebook uses synthetic/sample data for public portfolio demonstration.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


In [ ]:
file_path = "data/sample_lab_data.csv"

df = pd.read_csv(file_path)
df["Date"] = pd.to_datetime(df["Date"])
df["Qty"] = pd.to_numeric(df["Qty"], errors="coerce").fillna(0)

df = df.sort_values(["name", "Date"])

monthly = (
    df.groupby(["name", pd.Grouper(key="Date", freq="MS")])["Qty"]
    .sum()
    .reset_index()
    .sort_values(["name", "Date"])
)

print("Data loaded successfully")
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Lab tests:", monthly["name"].nunique())
print("Monthly rows:", len(monthly))


In [ ]:
def ets_forecast(train):
    train = train.astype(float)

    if len(train) >= 24:
        try:
            model = ExponentialSmoothing(
                train,
                trend="add",
                damped_trend=True,
                seasonal="add",
                seasonal_periods=12
            ).fit(optimized=True)

            return max(0, float(model.forecast(1).iloc[0]))
        except Exception:
            pass

    return max(0, float(train.tail(3).mean()))


In [ ]:
def create_xgb_data(series):
    data = pd.DataFrame({"y": series.values}, index=series.index)

    for lag in [1, 2, 3, 6, 12]:
        data[f"lag_{lag}"] = data["y"].shift(lag)

    data["rolling_3"] = data["y"].shift(1).rolling(3).mean()
    data["rolling_6"] = data["y"].shift(1).rolling(6).mean()
    data["rolling_12"] = data["y"].shift(1).rolling(12).mean()
    data["month"] = data.index.month
    data["trend"] = np.arange(len(data))

    return data.dropna()


def xgb_forecast(train):
    data = create_xgb_data(train)

    if len(data) < 12:
        return max(0, float(train.tail(3).mean()))

    X = data.drop(columns=["y"])
    y = data["y"]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X, y)

    future_date = train.index[-1] + pd.offsets.MonthBegin(1)
    future = pd.DataFrame(index=[future_date])

    for lag in [1, 2, 3, 6, 12]:
        future[f"lag_{lag}"] = train.iloc[-lag]

    future["rolling_3"] = train.tail(3).mean()
    future["rolling_6"] = train.tail(6).mean()
    future["rolling_12"] = train.tail(12).mean()
    future["month"] = future_date.month
    future["trend"] = len(train)

    return max(0, float(model.predict(future)[0]))


In [ ]:
def select_best_model(train):
    if len(train) < 30:
        return "ETS"

    validation_size = 6
    train_part = train.iloc[:-validation_size]
    actual = train.iloc[-validation_size:]

    ets_predictions = []
    xgb_predictions = []
    history = train_part.copy()

    for i in range(validation_size):
        ets_predictions.append(ets_forecast(history))
        xgb_predictions.append(xgb_forecast(history))

        history = pd.concat([
            history,
            pd.Series([actual.iloc[i]], index=[actual.index[i]])
        ])

    ets_mae = mean_absolute_error(actual, ets_predictions)
    xgb_mae = mean_absolute_error(actual, xgb_predictions)

    return "ETS" if ets_mae <= xgb_mae else "XGBoost"


In [ ]:
forecast_months = pd.date_range(
    start="2026-03-01",
    end="2026-10-01",
    freq="MS"
)

results = []

for test_name in monthly["name"].unique():
    test_data = monthly[monthly["name"] == test_name].copy()

    series = (
        test_data
        .set_index("Date")["Qty"]
        .sort_index()
        .asfreq("MS", fill_value=0)
    )

    training_history = series[series.index <= "2026-02-01"]
    best_model = select_best_model(training_history)

    for forecast_month in forecast_months:
        train = series[series.index < forecast_month]

        if best_model == "ETS":
            predicted = ets_forecast(train)
        else:
            predicted = xgb_forecast(train)

        actual = series.loc[forecast_month] if forecast_month in series.index else np.nan

        error_pct = (
            abs(actual - predicted) / actual * 100
            if pd.notna(actual) and actual != 0 else np.nan
        )

        results.append({
            "Lab Test": test_name,
            "Month": forecast_month,
            "Actual": actual,
            "Predicted": predicted,
            "Error %": error_pct,
            "Model": best_model
        })

results_df = pd.DataFrame(results)
results_df["Actual"] = results_df["Actual"].round(0)
results_df["Predicted"] = results_df["Predicted"].round(0)
results_df["Error %"] = results_df["Error %"].round(2)

print("Prediction completed")
print("Rows:", len(results_df))
results_df.head()


In [ ]:
summary = []

for test_name in results_df["Lab Test"].unique():
    temp = results_df[results_df["Lab Test"] == test_name]
    evaluated = temp.dropna(subset=["Actual"])

    mae = mean_absolute_error(evaluated["Actual"], evaluated["Predicted"])
    rmse = np.sqrt(mean_squared_error(evaluated["Actual"], evaluated["Predicted"]))

    summary.append({
        "Lab Test": test_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "Average Error %": round(evaluated["Error %"].mean(), 2),
        "Selected Model": temp["Model"].iloc[0]
    })

summary_df = pd.DataFrame(summary)
summary_df


In [ ]:
monthly_total = (
    results_df
    .groupby("Month")
    .agg(Actual=("Actual", "sum"), Predicted=("Predicted", "sum"))
    .reset_index()
)

monthly_total["Error %"] = (
    abs(monthly_total["Actual"] - monthly_total["Predicted"])
    / monthly_total["Actual"] * 100
)

monthly_total["Actual"] = monthly_total["Actual"].round(0)
monthly_total["Predicted"] = monthly_total["Predicted"].round(0)
monthly_total["Error %"] = monthly_total["Error %"].round(2)

monthly_total


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly_total["Month"], monthly_total["Actual"], marker="o", label="Actual")
plt.plot(monthly_total["Month"], monthly_total["Predicted"], marker="o", label="Predicted")
plt.title("Total Lab Demand: Actual vs Predicted")
plt.xlabel("Month")
plt.ylabel("Test Volume")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig("charts/actual_vs_predicted.png", dpi=150)
plt.show()


In [ ]:
model_counts = summary_df["Selected Model"].value_counts()

plt.figure(figsize=(7, 4))
model_counts.plot(kind="bar")
plt.title("Selected Model by Lab Test")
plt.xlabel("Model")
plt.ylabel("Number of Lab Tests")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("charts/model_comparison.png", dpi=150)
plt.show()


In [ ]:
results_df.to_csv("results/sample_forecast_results.csv", index=False)
summary_df.to_csv("results/model_summary.csv", index=False)
monthly_total.to_csv("results/monthly_total.csv", index=False)

print("Results saved successfully.")
